# Modelo 3: ECG Beat Classifier (signal-to-classification)

Modelo de clasificación **ECG real → diagnóstico por latido**, usando anotaciones
reales de cardiólogos (archivos `.atr` de MIT-BIH), siguiendo el estándar clínico AAMI.

Este es un **tercer modelo independiente** de `main.py` (clasificador por reglas) y
de `Modelo2.ipynb` (reconstrucción PPG→ECG). No usa PPG en ningún punto, por lo que
**no tiene el problema de data leakage** documentado en `Modelo2.ipynb`.

**Mejoras respecto a la primera versión:**

1. **Pesos de clase limitados (cap)** — en la primera corrida, las clases
   minoritarias (`S`, `F`) recibían pesos extremadamente altos por su rareza,
   lo que causaba que el modelo sobre-predijera esas clases (precisión de
   `S` = 0.04, F1 macro = 0.41). Ahora se limita el peso máximo relativo para
   evitar que una clase domine el gradiente.
2. **Checkpoint y reanudación de entrenamiento**, igual que en `Modelo2.ipynb`:
   si ya existe un checkpoint, la ejecución continúa desde ahí en vez de
   reiniciar desde cero.
3. **Registro histórico de resultados de test**: cada vez que corres el
   notebook hasta el final, se guarda el resultado y se compara
   automáticamente contra la corrida anterior, mostrando si hubo mejora o no.


## Instalaciones

In [ ]:
# %pip install torch wfdb scikit-learn
# %pip cache purge
# import sys
# print(sys.executable)

/home/jesus/VsCode/Proyectos/LectorBPM (Pruebas)/.venv/bin/python


## Imports

In [1]:
import os
import json
import random
import datetime
from pathlib import Path
from collections import Counter
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import wfdb
from scipy.signal import resample
from sklearn.metrics import classification_report, confusion_matrix, f1_score

## Utilidades

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

## Mapeo de anotaciones MIT-BIH → clases AAMI

Referencia: estándar AAMI EC57, ampliamente usado en literatura de clasificación
de arritmias sobre MIT-BIH.

In [3]:
AAMI_MAP = {
    # Normal
    'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
    # Supraventricular ectopic
    'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
    # Ventricular ectopic
    'V': 'V', 'E': 'V',
    # Fusion
    'F': 'F',
    # Unknown / paced
    '/': 'Q', 'f': 'Q', 'Q': 'Q',
}

CLASS_NAMES = ['N', 'S', 'V', 'F', 'Q']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

## Dataset: ventanas centradas en cada latido anotado

In [ ]:
@dataclass
class BeatWindowConfig:
    fs_target: int = 360
    window_sec: float = 1.0
    pre_r_sec: float = 0.4


class MITBIHBeatDataset(Dataset):
    def __init__(
        self,
        mit_path: str,
        split: str = "train",
        config: Optional[BeatWindowConfig] = None,
        train_ratio: float = 0.70,
        val_ratio: float = 0.15,
        seed: int = 42,
    ):
        self.cfg = config or BeatWindowConfig()
        self.win_len = int(self.cfg.window_sec * self.cfg.fs_target)
        self.pre_r = int(self.cfg.pre_r_sec * self.cfg.fs_target)

        mit_path = Path(mit_path)
        records = sorted({f.stem for f in mit_path.glob("*.dat")})
        if not records:
            raise FileNotFoundError(f"No se encontraron registros .dat en {mit_path}")

        rng = random.Random(seed)
        rng.shuffle(records)
        n = len(records)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        if split == "train":
            selected = records[:n_train]
        elif split == "val":
            selected = records[n_train:n_train + n_val]
        else:
            selected = records[n_train + n_val:]

        self.samples: List[Tuple[str, int, str]] = []  # (record, sample_idx, aami_class)

        for rec_name in selected:
            rec_path = str(mit_path / rec_name)
            try:
                ann = wfdb.rdann(rec_path, "atr")
            except Exception:
                continue

            for sample_idx, symbol in zip(ann.sample, ann.symbol):
                aami = AAMI_MAP.get(symbol)
                if aami is None:
                    continue  # simbolo no clinico (marcador de ritmo, etc.)
                self.samples.append((rec_name, int(sample_idx), aami))

        self._record_cache: Dict[str, np.ndarray] = {}
        self._record_fs: Dict[str, int] = {}
        self.mit_path = mit_path

        print(f"[{split:5s}] {len(selected)} registros -> {len(self.samples)} latidos anotados")
        dist = Counter(c for _, _, c in self.samples)
        print(f"          distribucion: {dict(dist)}")

    def _load_record(self, rec_name: str) -> Tuple[np.ndarray, int]:
        if rec_name not in self._record_cache:
            record = wfdb.rdrecord(str(self.mit_path / rec_name))
            signal = record.p_signal[:, 0].astype(np.float32)  # canal 0 (MLII habitualmente)
            fs = record.fs
            if fs != self.cfg.fs_target:
                new_len = int(len(signal) * self.cfg.fs_target / fs)
                signal = resample(signal, new_len).astype(np.float32)
            # normalizacion z-score por registro
            signal = (signal - signal.mean()) / (signal.std() + 1e-8)
            self._record_cache[rec_name] = signal
            self._record_fs[rec_name] = fs
        return self._record_cache[rec_name], self._record_fs[rec_name]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rec_name, sample_idx, aami = self.samples[idx]
        signal, orig_fs = self._load_record(rec_name)

        if orig_fs != self.cfg.fs_target:
            sample_idx = int(sample_idx * self.cfg.fs_target / orig_fs)

        start = sample_idx - self.pre_r
        end = start + self.win_len

        if start < 0 or end > len(signal):
            window = np.zeros(self.win_len, dtype=np.float32)
            src_start = max(0, start)
            src_end = min(len(signal), end)
            dst_start = src_start - start
            dst_end = dst_start + (src_end - src_start)
            window[dst_start:dst_end] = signal[src_start:src_end]
        else:
            window = signal[start:end]

        x = torch.tensor(window, dtype=torch.float32).unsqueeze(0)  # [1, win_len]
        y = torch.tensor(CLASS_TO_IDX[aami], dtype=torch.long)
        return x, y

## Modelo: CNN 1D + atención temporal + cabeza de clasificación

In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=7, stride=1, pool=2, dropout=0.1):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, stride=stride, padding=pad)
        self.bn = nn.BatchNorm1d(out_ch)
        self.pool = nn.MaxPool1d(pool) if pool > 1 else nn.Identity()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.relu(self.bn(self.conv(x)))
        x = self.pool(x)
        return self.dropout(x)


class TemporalAttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x):
        # x: [B, T, C]
        weights = torch.softmax(self.score(x).squeeze(-1), dim=1)  # [B, T]
        pooled = torch.einsum("bt,btc->bc", weights, x)
        return pooled, weights


class ECGBeatClassifier(nn.Module):
    def __init__(self, n_classes: int = len(CLASS_NAMES), dropout: float = 0.2):
        super().__init__()
        self.blocks = nn.Sequential(
            ConvBlock(1, 32, kernel_size=15, pool=2, dropout=dropout),
            ConvBlock(32, 64, kernel_size=11, pool=2, dropout=dropout),
            ConvBlock(64, 128, kernel_size=7, pool=2, dropout=dropout),
            ConvBlock(128, 128, kernel_size=5, pool=2, dropout=dropout),
        )
        self.attn_pool = TemporalAttentionPool(128)
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x, return_attention: bool = False):
        # x: [B, 1, T]
        z = self.blocks(x)             # [B, C, T']
        z = z.transpose(1, 2)          # [B, T', C]
        pooled, attn = self.attn_pool(z)
        logits = self.head(pooled)
        if return_attention:
            return logits, attn
        return logits

## Pérdida ponderada por clase, con límite (cap)

**Este es el cambio clave respecto a la primera versión.** MIT-BIH está muy
desbalanceado, así que ponderar por frecuencia inversa es necesario — pero sin
límite, las clases muy raras (`S`, `F`) reciben pesos absurdamente altos, lo que
hace que el modelo sobre-prediga esas clases para "ahorrarse" la pérdida
inflada, sacrificando precisión en todas las demás.

`CLASS_WEIGHT_CAP` limita cuánto más puede pesar la clase más rara frente a la
más común. Un valor típico razonable es entre 5 y 10.

In [6]:
CLASS_WEIGHT_CAP = 6.0

def compute_class_weights(dataset: MITBIHBeatDataset, cap: float = CLASS_WEIGHT_CAP) -> torch.Tensor:
    counts = Counter(c for _, _, c in dataset.samples)
    total = sum(counts.values())
    raw_weights = torch.zeros(len(CLASS_NAMES))
    for c, idx in CLASS_TO_IDX.items():
        n_c = counts.get(c, 1)
        raw_weights[idx] = total / (len(CLASS_NAMES) * n_c)

    # Limitar el peso maximo relativo al minimo, para que ninguna clase
    # domine el gradiente de forma desproporcionada.
    min_w = raw_weights.min()
    capped = torch.clamp(raw_weights, max=min_w * cap)

    # Renormalizar para que el promedio de los pesos sea 1 (escala estable)
    capped = capped / capped.mean()

    print("Pesos de clase (sin limitar) ->", {CLASS_NAMES[i]: round(w.item(), 2) for i, w in enumerate(raw_weights)})
    print("Pesos de clase (limitados)   ->", {CLASS_NAMES[i]: round(w.item(), 2) for i, w in enumerate(capped)})
    return capped

def make_balanced_sampler(dataset: MITBIHBeatDataset) -> WeightedRandomSampler:
    counts = Counter(c for _, _, c in dataset.samples)
    sample_weights = [
        1.0 / counts[c] for _, _, c in dataset.samples
    ]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(dataset),
        replacement=True,
    )

## Entrenamiento y evaluación

In [7]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        n += x.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(y.cpu().tolist())

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / n, f1


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        n += x.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(y.cpu().tolist())

    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / n, f1, all_preds, all_labels

## Configuración y carga de datos

In [8]:
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando device: {device}")

mit_path = "mit-bih-arrhythmia-database-1.0.0"

train_ds = MITBIHBeatDataset(mit_path, split="train")
val_ds = MITBIHBeatDataset(mit_path, split="val")
test_ds = MITBIHBeatDataset(mit_path, split="test")

train_sampler = make_balanced_sampler(train_ds)

train_loader = DataLoader(train_ds, batch_size=128, sampler=train_sampler, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=0)

Usando device: cpu
[train] 33 registros -> 76917 latidos anotados
          distribucion: {'F': 788, 'V': 6079, 'N': 65196, 'S': 982, 'Q': 3872}
[val  ] 7 registros -> 15659 latidos anotados
          distribucion: {'N': 11633, 'S': 1486, 'Q': 2091, 'V': 436, 'F': 13}
[test ] 8 registros -> 16918 latidos anotados
          distribucion: {'N': 13802, 'V': 721, 'F': 2, 'S': 313, 'Q': 2080}


## Instanciar modelo, pérdida y optimizador

In [9]:
model = ECGBeatClassifier().to(device)
print(f"Parametros: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

Parametros: 172,038


## Loop de entrenamiento con checkpoint y reanudación

In [ ]:
CKPT_DIR = "checkpoints_classifier"
CKPT_PATH = os.path.join(CKPT_DIR, "best_model.pt")
HISTORY_PATH = os.path.join(CKPT_DIR, "history.json")

SESSION_EPOCHS = 50
PATIENCE = 20

os.makedirs(CKPT_DIR, exist_ok=True)

start_epoch = 1
best_val_f1 = -1.0
history = []

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    if "optim_state" in ckpt:
        optimizer.load_state_dict(ckpt["optim_state"])
    if "scheduler_state" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt.get("epoch", 0) + 1
    best_val_f1 = ckpt.get("val_f1", -1.0)

    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH) as f:
            history = json.load(f)

    print(f"Checkpoint encontrado. Retomando desde epoch {start_epoch}.")
    print(f"Mejor val_f1 de la corrida anterior: {best_val_f1:.4f}")
else:
    print("No se encontro checkpoint previo. Entrenando desde cero.")

patience_counter = 0
end_epoch = start_epoch + SESSION_EPOCHS - 1

for epoch in range(start_epoch, end_epoch + 1):
    train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history.append({"epoch": epoch, "train_loss": train_loss, "train_f1": train_f1,
                     "val_loss": val_loss, "val_f1": val_f1})
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)

    print(f"Epoch {epoch:03d} | Train Loss={train_loss:.4f} F1={train_f1:.4f} "
          f"| Val Loss={val_loss:.4f} F1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save({
            "model_state": model.state_dict(),
            "optim_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss,
            "val_f1": val_f1,
        }, CKPT_PATH)
        print(f"  -> Guardado nuevo mejor modelo (val_f1={val_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping en epoch {epoch} (sin mejora en {PATIENCE} epocas)")
            break

Checkpoint encontrado. Retomando desde epoch 6.
Mejor val_f1 de la corrida anterior: 0.5736
Epoch 006 | Train Loss=0.1852 F1=0.9341 | Val Loss=0.9765 F1=0.5377
Epoch 007 | Train Loss=0.1317 F1=0.9525 | Val Loss=1.0930 F1=0.5107
Epoch 008 | Train Loss=0.1113 F1=0.9603 | Val Loss=1.2132 F1=0.5018
Epoch 009 | Train Loss=0.0973 F1=0.9659 | Val Loss=1.0028 F1=0.5395
Epoch 010 | Train Loss=0.0715 F1=0.9755 | Val Loss=1.1148 F1=0.5153
Epoch 011 | Train Loss=0.0669 F1=0.9777 | Val Loss=1.1976 F1=0.4825
Epoch 012 | Train Loss=0.0608 F1=0.9793 | Val Loss=1.1040 F1=0.5091
Epoch 013 | Train Loss=0.0585 F1=0.9801 | Val Loss=1.4302 F1=0.4570
Epoch 014 | Train Loss=0.0523 F1=0.9819 | Val Loss=1.3680 F1=0.4784
Epoch 015 | Train Loss=0.0492 F1=0.9836 | Val Loss=1.2954 F1=0.4848
Epoch 016 | Train Loss=0.0462 F1=0.9846 | Val Loss=1.4658 F1=0.4360
Epoch 017 | Train Loss=0.0451 F1=0.9853 | Val Loss=1.2567 F1=0.4928
Epoch 018 | Train Loss=0.0405 F1=0.9868 | Val Loss=1.3220 F1=0.4737
Epoch 019 | Train Loss=0

## Evaluación final en test set, con comparación contra la corrida anterior

In [63]:
TEST_LOG_PATH = os.path.join(CKPT_DIR, "test_runs.json")

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state"])
test_loss, test_f1, preds, labels = evaluate(model, test_loader, criterion, device)

report_dict = classification_report(labels, preds, target_names=CLASS_NAMES,
                                      zero_division=0, output_dict=True)
per_class_f1 = {c: round(report_dict[c]["f1-score"], 4) for c in CLASS_NAMES}

print("\n=== Evaluacion en test set (corrida actual) ===")
print(f"Epoch del checkpoint usado: {ckpt['epoch']}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 (macro): {test_f1:.4f}")
print("\nReporte de clasificacion:")
print(classification_report(labels, preds, target_names=CLASS_NAMES, zero_division=0))
print("\nMatriz de confusion:")
print(confusion_matrix(labels, preds))

# --- Cargar corridas anteriores y comparar ---
previous_runs = []
if os.path.exists(TEST_LOG_PATH):
    with open(TEST_LOG_PATH) as f:
        previous_runs = json.load(f)

current_run = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "epoch": ckpt["epoch"],
    "test_loss": round(test_loss, 4),
    "test_f1_macro": round(test_f1, 4),
    "per_class_f1": per_class_f1,
}

print("\n" + "=" * 60)
if previous_runs:
    last_run = previous_runs[-1]
    print("COMPARACION CON LA CORRIDA ANTERIOR")
    print("=" * 60)
    print(f"{'Metrica':<15}{'Anterior':>12}{'Actual':>12}{'Cambio':>12}")
    d_loss = current_run["test_loss"] - last_run["test_loss"]
    d_f1 = current_run["test_f1_macro"] - last_run["test_f1_macro"]
    print(f"{'Test Loss':<15}{last_run['test_loss']:>12.4f}{current_run['test_loss']:>12.4f}{d_loss:>+12.4f}")
    print(f"{'Test F1 macro':<15}{last_run['test_f1_macro']:>12.4f}{current_run['test_f1_macro']:>12.4f}{d_f1:>+12.4f}")
    print()
    for c in CLASS_NAMES:
        prev_f1 = last_run["per_class_f1"].get(c, 0.0)
        curr_f1 = current_run["per_class_f1"].get(c, 0.0)
        print(f"  F1 clase {c:<3}: {prev_f1:.4f} -> {curr_f1:.4f}  ({curr_f1 - prev_f1:+.4f})")

    if d_f1 > 0:
        print("\nEl modelo MEJORO respecto a la corrida anterior (F1 macro mas alto).")
    elif d_f1 < 0:
        print("\nEl modelo EMPEORO respecto a la corrida anterior (F1 macro mas bajo).")
    else:
        print("\nSin cambio respecto a la corrida anterior.")
else:
    print("No hay corridas anteriores registradas. Esta es la primera.")
print("=" * 60)

previous_runs.append(current_run)
with open(TEST_LOG_PATH, "w") as f:
    json.dump(previous_runs, f, indent=2)


=== Evaluacion en test set (corrida actual) ===
Epoch del checkpoint usado: 5
Test Loss: 1.0362
Test F1 (macro): 0.4401

Reporte de clasificacion:
              precision    recall  f1-score   support

           N       0.98      0.70      0.82     13802
           S       0.04      0.27      0.07       313
           V       0.24      0.90      0.38       721
           F       0.00      0.00      0.00         2
           Q       0.88      1.00      0.93      2080

    accuracy                           0.74     16918
   macro avg       0.43      0.57      0.44     16918
weighted avg       0.92      0.74      0.80     16918


Matriz de confusion:
[[9685 1854 1995    0  268]
 [ 121   83   84    0   25]
 [  48   26  647    0    0]
 [   2    0    0    0    0]
 [   0    0    2    0 2078]]

COMPARACION CON LA CORRIDA ANTERIOR
Metrica            Anterior      Actual      Cambio
Test Loss            1.0362      1.0362     +0.0000
Test F1 macro        0.4401      0.4401     +0.0000

  F1 c